In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import h5py
from pathlib import Path

In [2]:
base_dir      = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
transport_dir = base_dir / "Updated Code" / "transport narrowed param ranges"
mat_path = transport_dir / "all_trial_bkt_data.mat"

In [3]:
def load_all_trial_bkt_data(mat_path):
    param_sets = []
    with h5py.File(mat_path, 'r') as f:
        weights = f['weights'][()].T  # (num_param_sets, 3)
        atbd = f['all_trial_bkt_data']  # (1, N) object refs
        for i in range(atbd.shape[1]):
            ps_grp = f[atbd[0, i]]
            p0 = float(ps_grp['p0'][()].flat[0])
            td = ps_grp['trial_data']  # (1, 1000) object refs
            trial_data = []
            for j in range(td.shape[1]):
                arr = f[td[0, j]][()]
                if arr.dtype == np.float64:
                    trial_data.append(arr.flatten())
                else:
                    trial_data.append(np.array([], dtype=np.float64))
            param_sets.append({'p0': p0, 'trial_data': trial_data})
    return param_sets, weights


def recover_theta(w):
    disc = 1*w[0] + 2.1*(1 - w[0])       # [1, 2.1]
    b    = 0.01e-3*w[1] + 0.3e-3*(1 - w[1])  # [0.01e-3, 0.3e-3]
    lr   = 0.3*w[2] + 0.5*(1 - w[2])     # [0.3, 0.5]
    return disc, lr, b


param_sets, weights = load_all_trial_bkt_data(mat_path)
print(f"Loaded {len(param_sets)} param sets, {len(param_sets[0]['trial_data'])} trials each")

Loaded 20 param sets, 1000 trials each


In [4]:
# Quantiles and plot settings — adjust as needed after first inspection
# Np must match the particle count used when the data was generated
# (Updated Code/outputsT_at_varying_parameters.m), so that rank = floor(p * Np)
# refers to the correct particle out of all particles injected, not just the
# ones that happened to break through.
Np      = 1000
p_vals  = [0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9]
colors  = plt.cm.plasma(np.linspace(0.1, 0.9, len(p_vals)))
N_BINS  = 60
print('Quantiles:', p_vals)
print('Finest resolvable quantile given Np =', Np, 'is', 1 / Np)

Quantiles: [0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9]
Finest resolvable quantile given Np = 1000 is 0.001


In [5]:
def quantile_bkt(bkt, p, Np):
    """pth-percentile breakthrough time for one trial, or None if undefined
    (fewer than p*Np particles broke through in this trial)."""
    k = int(np.floor(p * Np))
    if k < len(bkt):
        return bkt[k]
    return None


def mean_bkt(bkt):
    """Mean breakthrough time among particles that broke through in this
    trial, or None if none did (disconnected trial)."""
    if len(bkt) == 0:
        return None
    return np.mean(bkt)

In [6]:
N_PARAM_SETS = 20
pdf_path = transport_dir / '[per quantile] distrib of pth percentile bkt.pdf'

with PdfPages(pdf_path) as pdf:
    for ps_idx in range(N_PARAM_SETS):
        ps = param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])

        fig, (ax_q, ax_m) = plt.subplots(1, 2, figsize=(14, 5))

        # --- left: distribution of the pth-percentile BKT, one curve per p ---
        for p, color in zip(p_vals, colors):
            vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
            vals = np.array([v for v in vals if v is not None])
            n_included = len(vals)
            if n_included < 2:
                continue

            log_vals = np.log10(vals)
            counts, edges = np.histogram(log_vals, bins=N_BINS, density=True)
            centres = (edges[:-1] + edges[1:]) / 2
            ax_q.plot(centres, counts, color=color,
                      label=f'p={p:.3f}  (n={n_included})')

        ax_q.set_xlabel('breakthrough time (s)')
        ax_q.set_ylabel('probability density')
        ticks = ax_q.get_xticks()
        ax_q.set_xticks(ticks)
        ax_q.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
        ax_q.set_title('Distribution of pth-percentile BKT across trials')
        ax_q.legend(fontsize=7)
        ax_q.grid(True, alpha=0.3)

        # --- right: distribution of the mean BKT per trial ---
        mvals = [mean_bkt(bkt) for bkt in ps['trial_data']]
        mvals = np.array([v for v in mvals if v is not None])
        if len(mvals) >= 2:
            log_mvals = np.log10(mvals)
            counts, edges = np.histogram(log_mvals, bins=N_BINS, density=True)
            centres = (edges[:-1] + edges[1:]) / 2
            ax_m.plot(centres, counts, color='tab:blue',
                      label=f'mean BKT  (n={len(mvals)})')

        ax_m.set_xlabel('breakthrough time (s)')
        ax_m.set_ylabel('probability density')
        ticks = ax_m.get_xticks()
        ax_m.set_xticks(ticks)
        ax_m.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
        ax_m.set_title('Distribution of mean BKT across trials')
        ax_m.legend(fontsize=7)
        ax_m.grid(True, alpha=0.3)

        fig.suptitle(
            f'Param set {ps_idx+1}  disc={disc:.2f}, lr={lr:.2f}, '
            f'b={b:.2e}, p0={ps["p0"]:.2f}')
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}')

/var/folders/zm/fpq2g01j3r3f3yfy0phwy54c0000gn/T/ipykernel_27923/1845362955.py:31: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax_q.legend(fontsize=7)
/var/folders/zm/fpq2g01j3r3f3yfy0phwy54c0000gn/T/ipykernel_27923/1845362955.py:50: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax_m.legend(fontsize=7)
/var/folders/zm/fpq2g01j3r3f3yfy0phwy54c0000gn/T/ipykernel_27923/1845362955.py:31: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax_q.legend(fontsize=7)
/var/folders/zm/fpq2g01j3r3f3yfy0phwy54c0000gn/T/ipykernel_27923/1845362955.py:50: UserWarning: No artists with labels found to put in legend.  Note that artists w

Saved /Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP/Updated Code/transport modified param ranges/[per quantile] distrib of pth percentile bkt.pdf
